In [1]:
# Tecnología
import json
import calendar
import pandas as pd
from sparky_bc import Sparky
import datetime as dt
from dateutil.relativedelta import relativedelta

# files lz conection
path_sparky_conf = '/Users/santlond/Documents/sparky_conf.json'

# Configurar conexión a LZ
with open(path_sparky_conf, 'rb') as JSON_lz_File:
    sp_config = json.loads(JSON_lz_File.read())
    
USER='santlond'
PASS=sp_config['ID']
DSN='IMPALA_PROD'
LOGDIR= 'logs'
# sparky = Sparky(username=USER, password=PASS, dsn=DSN, hostname="sbmdeblze004.bancolombia.corp")
sparky = Sparky(username=USER, password=PASS, dsn=DSN, hostname="sbmdeblze004.bancolombia.corp", spark_submit="spark3-submit")
 
# sparky = Sparky(username=USER, password=PASS, dsn=DSN)

helper = sparky.helper

/Users/santlond/Documents/venv_py39_odbc/lib/python3.9/site-packages/helper/helper.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2026-08-14 16:08:16 - [WARNING] - No se encontro la carpeta "/Users/santlond/Documents/mmm_tdc/logs" para guardar los logs


 ____  _____ __  __  ___ _____ _____ 
|  _ \| ____|  \/  |/ _ \_   _| ____|
| |_) |  _| | |\/| | | | || | |  _|  
|  _ <| |___| |  | | |_| || | | |___ 
|_| \_\_____|_|  |_|\___/ |_| |_____|
                                     
 ____  ____   _    ____  _  __
/ ___||  _ \ / \  |  _ \| |/ /
\___ \| |_) / _ \ | |_) | ' / 
 ___) |  __/ ___ \|  _ <| . \ 
|____/|_| /_/   \_\_| \_\_|\_\
                              



# Introducción

In [2]:
# Obtener número de tarjetas vendidas por digital

sql = f"""
WITH outcome AS
  (SELECT year(f_venta) AS YEAR,
          month(f_venta) AS mes,
          year(f_venta)*10000 + month(f_venta)*100 + 1 AS f_venta_ym,
          count(*) AS num_tdc_new
   FROM resultados_vdm.sabana_ventas_digitales
   WHERE YEAR BETWEEN 2020 AND 2026
     AND prod IN ('TDC')
     AND tipo_venta = 'Digital'
   GROUP BY 1,
            2,
            3
   ORDER BY f_venta_ym DESC)
SELECT YEAR,
       mes,
       f_venta_ym,
       num_tdc_new,
       sum(num_tdc_new) OVER (PARTITION BY YEAR
                              ORDER BY YEAR, mes) AS num_tdc_new_cumsum_ym
FROM outcome
ORDER BY f_venta_ym DESC;
"""
df = helper.obtener_dataframe(sql)

2026-08-14 16:08:21 - [INFO] - Transcurrido: 1786741702, Tiempo de Refresco = 1000


------------------------------------------------------------
  i    tipo    nombre    estado     hora_inicio   duracion   
------------------------------------------------------------
 1/1 DATAFRAME        descargando   04:08:21 PM             

2026-08-14 16:08:33 - [INFO] - 80 filas, 5 columnas, 00:10.6 consultando, 00:00.6 descargando, 00:00.0 convirtiendo


 1/1 DATAFRAME         finalizado   04:08:21 PM     00:11.6 
------------------------------------------------------------


In [3]:
# Ver datos
df

,year,mes,f_venta_ym,num_tdc_new,num_tdc_new_cumsum_ym
0,2026,8,20260801,17295,207402
1,2026,7,20260701,27951,190107
2,2026,6,20260601,26120,162156
3,2026,5,20260501,22204,136036
4,2026,4,20260401,30291,113832
...,...,...,...,...,...
75,2020,5,20200501,2734,26633
76,2020,4,20200401,401,23899
77,2020,3,20200301,8126,23498
78,2020,2,20200201,4948,15372


In [ ]:
# Escribir
df[['f_venta_ym', 'num_tdc_new']].to_excel('hist_data/ventas_tdc_digital_20240101_20260731.xlsx', index=False) # MODIFICAR nombre archivo. Cambia según fecha de ejecución.